In [3]:
from graphviz import Digraph

def create_panopticon_arch():
    dot = Digraph('MultiSensor_Panopticon', format='png')
    dot.attr(dpi='300', rankdir='TB', compound='true')
    dot.attr('node', fontname='Arial', shape='box', style='filled,rounded', fontsize='12')

    # --- 1. Multi-Sensor Input Stage (PE) ---
    with dot.subgraph(name='cluster_input') as c:
        c.attr(label='I. Sensor-Specific Patch Embedding', style='dashed', fontcolor='blue')
        sensors = [
            ('s2', 'Sentinel-2', '#CCEBC5'),
            ('l89', 'Landsat 8/9', '#B3CDE3'),
            ('s5p', 'TROPOMI (S5P)', '#DECBE4')
        ]
        for key, name, color in sensors:
            # 简化代码中的 Conv3d + ChnAttn + Proj
            c.node(f'PE_{key}', f'{name} PE\n(Conv3D + ChnAttn)', fillcolor=color)

    # --- 2. Shared Backbone (ViT Blocks) ---
    with dot.subgraph(name='cluster_backbone') as c:
        c.attr(label='II. Universal Backbone (DinoV2 Based)', style='filled', fillcolor='#F9F9F9')
        
        # 内部展示一个典型的 Block 结构，体现 DS-LN
        c.node('LN1', 'Domain-Specific LN 1\n(Selects s2/l89/s5p parameters)', fillcolor='#FFD1D1', color='red', penwidth='2')
        c.node('Attn', 'Shared Attention\n(Global Context)', fillcolor='#FFF2CC')
        c.node('LN2', 'Domain-Specific LN 2', fillcolor='#FFD1D1', color='red', penwidth='2')
        c.node('MLP', 'Shared SwiGLU FFN\n(Knowledge Base)', fillcolor='#FFF2CC')
        
        c.edge('LN1', 'Attn')
        c.edge('Attn', 'LN2')
        c.edge('LN2', 'MLP')
        
        # 标注 12层 堆叠
        c.node('Stack', '... Repeat x12 Blocks ...', shape='none', style='')
        c.edge('MLP', 'Stack')

    # --- 3. Output Heads ---
    with dot.subgraph(name='cluster_heads') as c:
        c.attr(label='III. Task-Specific Heads', style='dashed', fontcolor='green')
        for key, name, color in sensors:
            c.node(f'Head_{key}', f'CLS Head ({name})\nMethane Yes/No', fillcolor=color)

    # --- 连接全局 ---
    for key, _, _ in sensors:
        dot.edge(f'PE_{key}', 'LN1', lhead='cluster_backbone')
        dot.edge('Stack', f'Head_{key}', ltail='cluster_backbone')

    dot.render('panopticon_final_arch', cleanup=True)
    print("架构图已生成：panopticon_final_arch.png")

if __name__ == "__main__":
    create_panopticon_arch()

架构图已生成：panopticon_final_arch.png


In [4]:
from graphviz import Digraph

def create_residual_adapter_arch():
    dot = Digraph('Residual_Adapter_Panopticon', format='png')
    dot.attr(dpi='300', rankdir='TB', compound='true')
    dot.attr('node', fontname='Arial', shape='box', style='filled,rounded', fontsize='12')

    # --- 1. Sensor-Specific Input (PE) ---
    with dot.subgraph(name='cluster_input') as c:
        c.attr(label='I. Sensor-Specific Patch Embedding', style='dashed', fontcolor='#444444')
        sensors = [
            ('s2', 'Sentinel-2', '#CCEBC5'),
            ('l89', 'Landsat 8/9', '#B3CDE3'),
            ('s5p', 'S5P (TROPOMI)', '#DECBE4')
        ]
        for key, name, color in sensors:
            c.node(f'PE_{key}', f'{name} PE\n(3D Conv + ChnAttn)', fillcolor=color)

    # --- 2. Backbone with Residual Adapters ---
    with dot.subgraph(name='cluster_backbone') as c:
        c.attr(label='II. Hybrid ViT Backbone', style='filled', fillcolor='#F9F9F9')
        
        # 前 7 层：完全共享
        c.node('Blocks_Shared', '7 x Shared Transformer Blocks\n(Universal Spatial-Temporal Features)', 
               fillcolor='#FFF2CC', width='4')
        
        # 后 5 层：带 Adapter 的 Block
        with c.subgraph(name='cluster_adapter_blocks') as ab:
            ab.attr(label='5 x Sensor-Adapter Blocks', style='filled', fillcolor='#FFFFFF', color='#FF8C00')
            ab.node('Shared_Block', 'Shared Core Block\n(Attn + MLP)', fillcolor='#FFF2CC')
            
            # 并行的 Tiny Residual Adapters
            with ab.subgraph() as s:
                s.attr(rank='same')
                ab.node('Ad_S2', 'Tiny Adapter\n(S2 Specific)', fillcolor='#CCEBC5', style='filled,dashed')
                ab.node('Ad_L8', 'Tiny Adapter\n(L8/9 Specific)', fillcolor='#B3CDE3', style='filled,dashed')
                ab.node('Ad_S5P', 'Tiny Adapter\n(S5P Specific)', fillcolor='#DECBE4', style='filled,dashed')
            
            ab.edge('Shared_Block', 'Ad_S2', style='dotted', label='Residual')
            ab.edge('Shared_Block', 'Ad_L8', style='dotted')
            ab.edge('Shared_Block', 'Ad_S5P', style='dotted')

    # --- 3. Classifier Heads ---
    with dot.subgraph(name='cluster_heads') as c:
        c.attr(label='III. Specialized CLS Heads', style='dashed')
        for key, name, color in sensors:
            c.node(f'Head_{key}', f'CLS Head ({name})', fillcolor=color)

    # --- 连线 ---
    for key, _, _ in sensors:
        dot.edge(f'PE_{key}', 'Blocks_Shared')
    
    dot.edge('Blocks_Shared', 'Shared_Block')
    
    dot.edge('Ad_S2', 'Head_s2')
    dot.edge('Ad_L8', 'Head_l89')
    dot.edge('Ad_S5P', 'Head_s5p')

    dot.render('residual_adapter_arch', cleanup=True)
    print("架构图已生成：residual_adapter_arch.png")

if __name__ == "__main__":
    create_residual_adapter_arch()

架构图已生成：residual_adapter_arch.png
